Name:[Sinya]
Student ID: [023-24-0227]
Section: [G]
GitHub Profile: [https://github.com/sinyaakumarii/lab2_sinya_churn_.git]
Kaggle Profile: [https://www.kaggle.com/sinyakumari1]
Dataset: Telco Customer Churn
Dataset Source: [kaggale]

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
df = pd.read_csv('clean_churn.csv')

In [ ]:
y= df['Churn'].map({"Yes":1,"No":0})
X = df.drop(columns=['Churn'])
X= pd.get_dummies(X,drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 1. Decision Tree Baseline

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    max_depth=6,
    random_state=42
)

dt_model.fit(X_train, y_train)
dt_model.fit(X_train, y_train)

STEP 5 — Make Decision Tree predictions

In [ ]:
y_pred_dt = dt_model.predict(X_test)

print("First 20 predictions:")
print(y_pred_dt[:20])

STEP 6 — Calculate the four required metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt)
dt_recall = recall_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)

print("Decision Tree Baseline")
print("----------------------")
print(f"Accuracy : {dt_accuracy:.4f}")
print(f"Precision: {dt_precision:.4f}")
print(f"Recall   : {dt_recall:.4f}")
print(f"F1-score : {dt_f1:.4f}")

## Decision Tree Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm_dt = confusion_matrix(y_test, y_pred_dt)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_dt,
    display_labels=['No Churn', 'Churn']
)

disp.plot()
plt.title("Decision Tree Confusion Matrix")
plt.show()

STEP 8 — Classification Report


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_dt,
    target_names=['No Churn', 'Churn']
))

### Interpretation

the decision tree with max_depth=6 is used as the baseline model for this lab
accuracy measures the overall percentage of correct predictions, while
precision, recall, and F1-score provide more detailed information about the
churn class. because the Telco Churn dataset has class imbalance, recall and
F1-score for the Churn class are particularly important. The confusion matrix
also shows the number of customers correctly and incorrectly classified in
each class.

In [ ]:
#random forest
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

In [ ]:
#random forest predictions
y_pred_rf = rf_model.predict(X_test)

print("First 20 Random Forest predictions:")
print(y_pred_rf[:20])

In [ ]:
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print("Random Forest")
print("-------------")
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1-score : {rf_f1:.4f}")

In [ ]:
#confusion matrix of random forest
cm_rf = confusion_matrix(y_test, y_pred_rf)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=['No Churn', 'Churn']
)

disp.plot()
plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
#compare both models
comparison_df = pd.DataFrame({
    'Model': [
        'Decision Tree',
        'Random Forest'
    ],
    'Accuracy': [
        dt_accuracy,
        rf_accuracy
    ],
    'Precision': [
        dt_precision,
        rf_precision
    ],
    'Recall': [
        dt_recall,
        rf_recall
    ],
    'F1-score': [
        dt_f1,
        rf_f1
    ]
})

comparison_df

### Interpretation

the random forest achieved slightly higher accuracy and precision than the
Decision Tree. However, its recall and F1-score were lower. Since the dataset
is imbalanced and identifying customers who actually churn is important,
recall and F1-score are important evaluation measures. Therefore, the Random
Forest does not provide a clear overall improvement over the Decision Tree at
this stage.

### Why Cross-Validation?

a single train-test split can give results that depend on how the data was
divided. Five-fold cross-validation evaluates the model on five different
validation folds and provides a mean and standard deviation of the F1-score.
This gives a more reliable estimate of model performance.

In [ ]:
#cross validation for decision tree
from sklearn.model_selection import cross_val_score
dt_cv_scores = cross_val_score(
    dt_model,
    X_train,
    y_train,
    cv=5,
    scoring='f1'
)

print("Decision Tree 5-Fold F1 Scores:")
print(dt_cv_scores)

print("\nMean F1-score:", dt_cv_scores.mean())
print("Standard Deviation:", dt_cv_scores.std())


In [ ]:
dt_cv_scores.mean()
dt_cv_scores.std()

In [ ]:
#cross validation for random forest
rf_cv_scores = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=5,
    scoring='f1'
)

print("Random Forest 5-Fold F1 Scores:")
print(rf_cv_scores)

print("\nMean F1-score:", rf_cv_scores.mean())
print("Standard Deviation:", rf_cv_scores.std())

In [ ]:
scoring='f1'

In [ ]:
#csv comparision table
cv_comparison = pd.DataFrame({
    'Model': [
        'Decision Tree',
        'Random Forest'
    ],
    'Mean F1': [
        dt_cv_scores.mean(),
        rf_cv_scores.mean()
    ],
    'Std F1': [
        dt_cv_scores.std(),
        rf_cv_scores.std()
    ]
})

cv_comparison

### Cross-Validation Interpretation

Five-fold cross-validation was used to obtain a more reliable estimate of
model performance. The Decision Tree achieved a mean F1-score of 0.5651 with
a standard deviation of 0.0391, while the Random Forest achieved a mean
F1-score of 0.5553 with a standard deviation of 0.0223.

The Decision Tree has the higher mean F1-score, indicating slightly better
average performance. The Random Forest has a smaller standard deviation,
which indicates more consistent performance across the five folds. Overall,
the cross-validation results are consistent with the single train/test split,
where the Decision Tree also had a higher F1-score. Therefore, the
cross-validation results support the conclusion that the Decision Tree
performs slightly better based on F1-score.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

rf_for_tuning = RandomForestClassifier(
    random_state=42
)

grid_search = GridSearchCV(
    estimator=rf_for_tuning,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Hyperparameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validated F1-score:")
print(grid_search.best_score_)

best_rf_model = grid_search.best_estimator_

print("\nBest Random Forest Model:")
print(best_rf_model)

In [ ]:
print("Best Hyperparameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validated F1-score:")
print(f"{grid_search.best_score_:.4f}")

best_rf_model = grid_search.best_estimator_

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

y_pred_tuned_rf = best_rf_model.predict(X_test)

tuned_rf_accuracy = accuracy_score(
    y_test,
    y_pred_tuned_rf
)

tuned_rf_precision = precision_score(
    y_test,
    y_pred_tuned_rf
)

tuned_rf_recall = recall_score(
    y_test,
    y_pred_tuned_rf
)

tuned_rf_f1 = f1_score(
    y_test,
    y_pred_tuned_rf
)

# Print results
print("Tuned Random Forest Results")
print("---------------------------")
print(f"Accuracy : {tuned_rf_accuracy:.4f}")
print(f"Precision: {tuned_rf_precision:.4f}")
print(f"Recall   : {tuned_rf_recall:.4f}")
print(f"F1-score : {tuned_rf_f1:.4f}")

cm_tuned_rf = confusion_matrix(
    y_test,
    y_pred_tuned_rf
)

print("\nConfusion Matrix:")
print(cm_tuned_rf)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_tuned_rf,
    display_labels=["No Churn", "Churn"]
)

disp.plot()
plt.title("Tuned Random Forest Confusion Matrix")
plt.show()

In [ ]:

final_comparison = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Default Random Forest",
        "Tuned Random Forest"
    ],
    "Accuracy": [
        dt_accuracy,
        rf_accuracy,
        tuned_rf_accuracy
    ],
    "Precision": [
        dt_precision,
        rf_precision,
        tuned_rf_precision
    ],
    "Recall": [
        dt_recall,
        rf_recall,
        tuned_rf_recall
    ],
    "F1-score": [
        dt_f1,
        rf_f1,
        tuned_rf_f1
    ]
})

print(final_comparison.to_string(index=False))

print("\nRounded Comparison:")
display(
    final_comparison.round(4)
)

In [ ]:
best_model_name = final_comparison.loc[
    final_comparison["F1-score"].idxmax(),
    "Model"
]

best_f1_value = final_comparison["F1-score"].max()

print("Selected Final Model:", best_model_name)
print(f"Best F1-score: {best_f1_value:.4f}")

### Task 14 — Interpretation

The final model is selected primarily using F1-score because the dataset has
class imbalance and both precision and recall are important. The selected model
achieves the highest F1-score among the Decision Tree, default Random Forest,
and tuned Random Forest. The decision is therefore based on held-out test-set
performance rather than training performance alone.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Select final model object
if best_model_name == "Decision Tree":
    final_model = dt_model

elif best_model_name == "Default Random Forest":
    final_model = rf_model

else:
    final_model = best_rf_model

# Get feature importance
feature_importance = pd.Series(
    final_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

# Display top 10 features
print("Top 10 Most Important Features:")
display(feature_importance.head(10))

# Plot top 10 features
top_features = feature_importance.head(10).sort_values()

plt.figure(figsize=(8, 5))

plt.barh(
    top_features.index,
    top_features.values
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 10 Feature Importances")

plt.show()

### Task 15 — Interpretation

The feature-importance analysis shows which variables contribute most strongly
to the final model's predictions. The most important features should be compared
with the findings from Lab 2 EDA and the Decision Tree in Lab 3. If features such
as tenure, Internet service type, monthly charges, or total charges appear among
the most important variables, this is consistent with the earlier analysis.

In [ ]:
final_predictions = final_model.predict(X_test)

if 'customerID' in df.columns:
    ids = df.loc[X_test.index, 'customerID']
else:
    ids = X_test.index

submission = pd.DataFrame({
    'customerID': ids,
    'Churn': final_predictions
})
submission['Churn'] = submission['Churn'].map({1: 'Yes', 0: 'No'})

submission.to_csv('submission.csv', index=False)
print(submission.head())
print(f"\nSubmission saved: {submission.shape[0]} rows")
